# Background arms: does the model still need the scenery?

The milestone 3 review found that our species model leans on the scenery around the snake. Paste a snake onto the scenery of a photo of another species and its top-1 falls from 0.628 to 0.460; paint the snake out and leave only the scenery, and it still gets 0.128 right. This notebook trains three ways of fixing that, the background **arms**, and scores each one with the same robustness test, so a fixed rule can pick the one we train on all the data.

| Arm | What the model sees |
|---|---|
| A0 | the detector's box crop: milestone 3's final model, not retrained |
| A1 | the box crop, with the snake pasted onto another species' scenery half the time in training |
| A2 | the snake only: everything outside the predicted mask filled grey, in training and in evaluation |
| A3 | a crop to the predicted mask's box, scenery swapped half the time in training |

Every arm trains on the experiment subset (13,925 photos) with the final model's recipe: 20 epochs, seed 1, the same learning rate, margin and augmentation. One arm runs per Kaggle session, chosen by `ARM` below. `ARM = "A0"` scores milestone 3's final model with this notebook's code, the check that the robustness test still gives milestone 3's numbers; `"A1"`, `"A2"` or `"A3"` trains that arm (about an hour and a half on a T4) and scores it.

It reads the cleaned photos, the SAM snake masks, the experiment subset and the front end's predictions (a box and a predicted mask for every photo, from the cropping notebook's Kaggle run); A0 also reads milestone 3's final run. Each run's folder lands under `/kaggle/working`, with the same files as every training run plus `decision_metrics.json` and `robustness_metrics.json`.

This copy has no outputs. The run of the shipped arm is `results/m4/arms/executed/18_background_arms_A3.ipynb`; the other arms' runs are beside it.

## Setup

The library versions the recorded runs used, and the project's code.

In [ ]:
%pip install -q timm==1.0.31 albumentations==2.0.8 scikit-learn==1.9.1
import hashlib, json, os, shutil, sys, warnings
from pathlib import Path

os.environ["NO_ALBUMENTATIONS_UPDATE"] = "1"      # no "new version available" check
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"  # no download bar for the ImageNet weights
warnings.filterwarnings("ignore")

import kagglehub, torch, yaml
import pandas as pd

REPO = Path("..").resolve()
sys.path.insert(0, str(REPO / "src"))  # the project's code
from serpentguard import data, robustness
from serpentguard.decide import decision_report
from serpentguard.metrics import write_json
from serpentguard.train import subset_hash, train

pd.set_option("display.precision", 3)
print(f"{torch.cuda.device_count()} GPU(s)")

## The arm

`ARM` picks what this session runs. `LIMIT` and `EPOCHS` are for a quick smoke run (a few hundred photos per set, one epoch); the real runs leave them at `None`. `FRONT_END` names where the front end's predictions come from: the cropping notebook's Kaggle run on our own account, or a private, byte-identical copy of it on a teammate's account, which can't read that run. `RUN_FINAL` does the same for milestone 3's final run, which only A0 reads.

In [ ]:
ARM = "A0"
LIMIT = None
EPOCHS = None
FRONT_END = "kernel:puneetbajajiitm/sg-m3-frontends"
RUN_FINAL = "puneetbajajiitm/serpentguard-m3-run-final"

## The settings

Every arm starts from the settings of milestone 3's final model, saved as `H1_top1_s1` (they are the training notebook's settings, copied here from the run's `config.yaml`), and changes only its front end and how often its scenery is swapped.

Which masks? A1's input is the detector's box crop, so its masks only cut out the snake for the swap, and for that we use the SAM masks, the best masks we have. A2 and A3 cut their input from the mask itself, so they train on the U-Net's predicted masks: those are the only masks the app has, and a model trained on cleaner masks than it will see would be scored on something it never sees in use. The validation photos only have predicted masks, so every arm is scored on those.

In [ ]:
H1 = {'run_id': 'H1_top1_s1',
 'encoder': 'mobilevit_s',
 'head': 'arcface',
 'losses': ['arcface'],
 'front_end': 'F1',
 'input_size': 320,
 'epochs': 20,
 'lr': 0.0009530320008110496,
 'weight_decay': 0.0015566936015792908,
 'batch_size': 64,
 'sampler': 'weighted',
 'seed': 1,
 'embedding_dim': 256,
 'arcface_s': 30.0,
 'arcface_m': 0.35616842912779545,
 'triplet_margin': 0.2,
 'triplet_weight': 1.0,
 'mask_weight': 1.0,
 'hierarchy': True,
 'mask_head': False,
 'mask_keep': 0.2,
 'pooling': 'gap',
 'danger_weight': 1.0,
 'aug_strength': 0.7454942047909925,
 'mask_source': 'predicted',
 'box_source': 'predicted',
 'eval_every': 2,
 'warmup_epochs': 1,
 'exclude_species': []}

TRAINED = {  # each arm's changes to H1
    "A1": {"front_end": "F1", "scenery_swap": 0.5, "mask_source": "sam"},
    "A2": {"front_end": "F3", "scenery_swap": 0.0},
    "A3": {"front_end": "F1m", "scenery_swap": 0.5},
}
BANK_FROM = {"front_end": "F1", "scenery_swap": 0.5}  # the bank's photos: H1's predicted masks
SUBSET_HASH = "40196d19a68e"  # the first 12 digits of the committed subset's SHA-256
FRONT_END_SHA256 = "9dcc0576eb1c60d3e767ba9ca3c34a6a1b92b6b01946e377042dee1b1287b9e3"
# milestone 3's robustness test of its final model (results/m3/r1/H1_top1_s1/robustness_metrics.json)
M3_TOP1 = {"blanked": 0.12755836514990299, "swapped": 0.4601614821305627}
CHECK_TOLERANCE = 0.002
assert ARM in ("A0", *TRAINED), f"unknown ARM {ARM!r}"

## The inputs

Found by handle with `kagglehub`. The front end's file must be byte-identical to milestone 3's, whichever account reads it.

In [ ]:
ON_KAGGLE = Path("/kaggle/working").exists()
OWNER = "puneetbajajiitm"
KERNEL = "kernel:"
WORK = Path("/kaggle/working") if ON_KAGGLE else REPO / "runs" / "m4_arms"


def dataset(slug):
    return Path(kagglehub.dataset_download(f"{OWNER}/{slug}"))


def download(handle):
    if handle.startswith(KERNEL):
        return Path(kagglehub.notebook_output_download(handle[len(KERNEL):]))
    return Path(kagglehub.dataset_download(handle))


def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


DATA = dataset("serpentguard-indian-snakes-clean")
MASKS = dataset("serpentguard-sam-masks")
SUBSET = dataset("serpentguard-m3-subset") / "subset.csv"
PREDS = next(download(FRONT_END).rglob("frontend_preds.csv")).parent

assert subset_hash(pd.read_csv(SUBSET)["path"]).startswith(SUBSET_HASH), "not the experiment subset"
assert sha256(PREDS / "frontend_preds.csv") == FRONT_END_SHA256, "not milestone 3's front end"
preds = pd.read_csv(PREDS / "frontend_preds.csv")
listed = preds["mask_file"].fillna("").loc[lambda masks: masks.ne("")]
assert all((PREDS / name).is_file() for name in listed), "a listed mask is missing"
print(f"front end: {preds['box_conf'].notna().sum():,} boxes and {len(listed):,} masks "
      f"for {len(preds):,} photos")

Each arm's settings, as `train()` will get them:

In [ ]:
def arm_config(arm):
    cfg = {**H1, "run_id": f"M4_{arm}", **TRAINED[arm]}
    return cfg if EPOCHS is None else {**cfg, "epochs": EPOCHS}


changed = ["run_id", "front_end", "scenery_swap", "mask_source", "box_source", "seed", "epochs"]
pd.DataFrame({arm: {key: arm_config(arm).get(key) for key in changed} for arm in TRAINED})

## The scenery bank

A swap pastes the snake onto the scenery of a training photo of another species. That scenery must hold no snake of its own, so the swapping arms first build a bank: 3,000 training photos with a usable predicted mask, the species taking turns with a fixed seed, each with its snake painted out by OpenCV inpainting (the mask grown by 5 pixels first) and shrunk to at most 640 pixels a side. The bank is always made with the predicted masks, so A1 and A3 build the same bank; its `bank.csv` fingerprint (SHA-256) is printed so the two runs can be compared. The evaluation swap below blanks the donor's snake grey instead: the two ways of removing a snake differ on purpose, so a model can't score well by learning the marks one of them leaves.

The photos may not be redistributed, so this notebook shows counts only.

In [ ]:
SWAPS = ARM in TRAINED and TRAINED[ARM]["scenery_swap"] > 0
BANK = WORK / "scenery_bank"
if SWAPS:
    frames = data.make_frames({**H1, "run_id": "scenery_bank", **BANK_FROM}, DATA, MASKS, SUBSET,
                              PREDS, LIMIT)
    bank = data.build_scenery_bank(frames["train"], BANK)
    per_class = bank["class_id"].value_counts()
    print(f"{len(bank):,} sceneries from {len(frames['train']):,} training photos "
          f"({frames['train']['mask_valid'].sum():,} with a usable mask), {len(per_class)} species, "
          f"{per_class.min()} to {per_class.max()} each; bank.csv sha256 {sha256(BANK / 'bank.csv')}")

## Training the arm

`train()` trains the arm and writes its run folder, exactly as for every training run in milestone 3. A0 is not retrained: its run folder is a copy of milestone 3's final run, because the next two steps write their results next to the files they read. For A0 we also check that the settings above are the ones that run was trained with.

In [ ]:
if ARM in TRAINED:
    cfg = {**arm_config(ARM), **({"scenery_bank": str(BANK)} if SWAPS else {})}
    RUN = WORK / cfg["run_id"]
    final = train(cfg, RUN, DATA, MASKS, SUBSET, preds_root=PREDS, limit=LIMIT)
else:
    M3_FINAL = download(RUN_FINAL) / "H1_top1_s1"
    saved = yaml.safe_load((M3_FINAL / "config.yaml").read_text())
    assert H1 == {key: value for key, value in saved.items() if key != "subset_path"}, "not H1"
    RUN = WORK / M3_FINAL.name
    shutil.copytree(M3_FINAL, RUN, dirs_exist_ok=True)
    final = json.loads((RUN / "final_metrics.json").read_text())
print(f"{RUN.name}: macro-F1 {final['head']['macro_f1']:.3f}, top-1 {final['head']['top1']:.3f}, "
      f"best epoch {final['run']['best_epoch']}")

## Verdicts and WHO recall

The safety layer's thresholds are refitted on this arm's own calibration photos, with the current rule (DR0) and the lower-bound fit, exactly as the safety notebook fits them for the final model. It also gets the same two extra inputs: each photo's dark and blurry flags, and the detector's box confidence. The number the pick reads is the **WHO recall**: the share of WHO-listed validation photos called dangerous.

In [ ]:
quality = robustness.quality_table(RUN, DATA)
box_conf = dict(zip(preds["path"], preds["box_conf"]))
decision = decision_report(RUN, DATA, quality=quality, box_conf=box_conf, recall_bound="lower")
verdicts = decision["verdict_metrics"]
print(f"WHO recall {verdicts['who_dangerous_recall']:.3f}, false alarms "
      f"{verdicts['false_alarm']:.3f}, we don't know {verdicts['abstain_rate']:.1%}")

## The robustness test

The safety notebook's test, now in `robustness.py`, run on every validation photo with a usable predicted mask:

- **snake blanked**: the snake's pixels painted grey, everything else kept;
- **scenery swapped**: the snake's pixels pasted, unchanged, onto the scenery of a validation photo of another species, drawn with a fixed seed (that photo's own snake blanked first).

For the arms that cut their input from the mask (A2 and A3), the photo's own mask still sets the crop and the grey outside, so A2 sees almost nothing of a blanked photo and almost the same pixels when the scenery is swapped. Before blanking anything, the as-is pass must give the run's own saved validation outputs back, at least 99% the same top species.

In [ ]:
(RUN / "r1").mkdir(exist_ok=True)
r1 = write_json(RUN / "robustness_metrics.json",
                robustness.robustness_metrics(RUN, DATA, PREDS, subset_csv=SUBSET, quality=quality,
                                              out_dir=RUN / "r1", limit=LIMIT))
c = r1["conditions"]
pd.Series({"validation photos": r1["photos"]["val"], "with a usable mask": r1["photos"]["with_mask"],
           "as-is agreement with the saved run": r1["instrument_check"]["top1_agreement"],
           "top-1 as is": c["as_is"]["top1"],
           "top-1 as is, photos with a mask": c["swapped"]["as_is_same_photos"]["top1"],
           "top-1 snake blanked": c["blanked"]["top1"], "top-1 scenery swapped": c["swapped"]["top1"],
           "follows the donor's species": c["swapped"]["donor"]["follows_donor"],
           "minutes": r1["seconds"] / 60}).to_frame(RUN.name)

### The check: milestone 3's numbers again

With `ARM = "A0"` the test runs on milestone 3's final model, so it must give milestone 3's results back: top-1 with the snake blanked and with the scenery swapped within 0.002 of the saved ones, and the same WHO recall as the decision file the app ships with.

In [ ]:
if ARM == "A0":
    shipped = json.loads((REPO / "results/m3/final/H1_top1_s1/decision_metrics.json").read_text())
    now = {"blanked": c["blanked"]["top1"], "swapped": c["swapped"]["top1"],
           "WHO recall": verdicts["who_dangerous_recall"]}
    before = {**M3_TOP1, "WHO recall": shipped["verdict_metrics"]["who_dangerous_recall"]}
    check = pd.DataFrame({"milestone 3": before, "this notebook": now})
    check["difference"] = check["this notebook"] - check["milestone 3"]
    assert check["difference"].abs().max() <= CHECK_TOLERANCE, check
    display(check.round(4))

## The arm's row

What the pick reads: normal macro-F1 (over all validation photos, from training), WHO recall, and top-1 on the photos with a usable mask, as they are and with the scenery swapped. The rule, applied once every arm has run, lets an arm win only if it leans on the scenery less than A0 does: an arm qualifies if its macro-F1 is at least A0's minus 0.005 and its WHO recall at least A0's minus 0.006 (the spread across milestone 3's three seeds of the final recipe). Of A0 and the qualifying arms, the smallest gap between normal and swapped top-1 wins, and on a tie the higher macro-F1. So an arm replaces A0 only with a smaller gap than A0's 0.168.

In [ ]:
row = robustness.arm_row(ARM, final, decision, r1)
row["gap"] = row["top1"] - row["top1_swapped"]
if SWAPS:
    row["bank_sha256"] = sha256(BANK / "bank.csv")
write_json(RUN / "arm_row.json", row)
pd.Series(row).to_frame(ARM)